In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

def llm(prompt):
    return w.serving_endpoints.query(
        name="databricks-meta-llama-3-1-8b-instruct",
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)]
    ).choices[0].message.content

def ask_reviews(question, k=5):
    res = w.vector_search_indexes.query_index(
        index_name="northstar.gold.reviews_index",
        columns=["review_comment_message"],
        query_text=question,
        num_results=k)
    reviews = "\n".join(f"- {row[0]}" for row in res.result.data_array)

    prompt = f"""You are a customer insights analyst.
Based ONLY on these customer reviews (they are in Portuguese), answer the question in English.

Reviews:
{reviews}

Question: {question}"""

    return llm(prompt)

def evaluate_answer(question, answer):
    prompt = f"""You are a strict evaluator. Score this AI answer 1-5 on:
- Relevance: does it actually answer the question?
- Groundedness: does it sound based on real customer reviews (not invented)?

Question: {question}
AI Answer: {answer}

Reply exactly as:
Score: <1-5>
Reason: <one sentence>"""
    return llm(prompt)

# Get an answer from your RAG bot, then judge it
q = "What are the main delivery complaints?"
a = ask_reviews(q)
print("ANSWER:\n", a)
print("\nJUDGE:\n", evaluate_answer(q, a))